# Your First RAG Application

In this notebook, we'll walk you through each of the components that are involved in a simple RAG application.

We won't be leveraging any fancy tools, just the OpenAI Python SDK, Numpy, and some classic Python.

> NOTE: This was done with Python 3.12.3.

> NOTE: There might be [compatibility issues](https://github.com/wandb/wandb/issues/7683) if you're on NVIDIA driver >552.44 As an interim solution - you can rollback your drivers to the 552.44.

## Table of Contents:

- Task 1: Imports and Utilities
- Task 2: Documents
- Task 3: Embeddings and Vectors
- Task 4: Prompts
- Task 5: Retrieval Augmented Generation
  - 🚧 Activity #1: Augment RAG

Let's look at a rather complicated looking visual representation of a basic RAG application.

<img src="https://i.imgur.com/vD8b016.png" />

## Task 1: Imports and Utility

We're just doing some imports and enabling `async` to work within the Jupyter environment here, nothing too crazy!

In [54]:
from aimakerspace.text_utils import TextFileLoader, CharacterTextSplitter
from aimakerspace.vectordatabase import VectorDatabase
import asyncio

In [55]:
import nest_asyncio
nest_asyncio.apply()

## Task 2: Documents

We'll be concerning ourselves with this part of the flow in the following section:

<img src="https://i.imgur.com/jTm9gjk.png" />

### Loading Source Documents

So, first things first, we need some documents to work with.

While we could work directly with the `.txt` files (or whatever file-types you wanted to extend this to) we can instead do some batch processing of those documents at the beginning in order to store them in a more machine compatible format.

In this case, we're going to parse our text file into a single document in memory.

Let's look at the relevant bits of the `TextFileLoader` class:

```python
def load_file(self):
        with open(self.path, "r", encoding=self.encoding) as f:
            self.documents.append(f.read())
```

We're simply loading the document using the built in `open` method, and storing that output in our `self.documents` list.

> NOTE: We're using blogs from PMarca (Marc Andreessen) as our sample data. This data is largely irrelevant as we want to focus on the mechanisms of RAG, which includes out data's shape and quality - but not specifically what the contents of the data are. 


In [56]:
text_loader = TextFileLoader("data/PMarcaBlogs.txt")
documents = text_loader.load_documents()
len(documents)

1

In [57]:
print(documents[0][:100])


The Pmarca Blog Archives
(select posts from 2007-2009)
Marc Andreessen
copyright: Andreessen Horow


### Splitting Text Into Chunks

As we can see, there is one massive document.

We'll want to chunk the document into smaller parts so it's easier to pass the most relevant snippets to the LLM.

There is no fixed way to split/chunk documents - and you'll need to rely on some intuition as well as knowing your data *very* well in order to build the most robust system.

For this toy example, we'll just split blindly on length.

>There's an opportunity to clear up some terminology here, for this course we will be stick to the following:
>
>- "source documents" : The `.txt`, `.pdf`, `.html`, ..., files that make up the files and information we start with in its raw format
>- "document(s)" : single (or more) text object(s)
>- "corpus" : the combination of all of our documents

As you can imagine (though it's not specifically true in this toy example) the idea of splitting documents is to break them into managable sized chunks that retain the most relevant local context.

In [58]:
text_splitter = CharacterTextSplitter()
split_documents = text_splitter.split_texts(documents)
len(split_documents)

373

Let's take a look at some of the documents we've managed to split.

In [59]:
split_documents[0:1]

['\ufeff\nThe Pmarca Blog Archives\n(select posts from 2007-2009)\nMarc Andreessen\ncopyright: Andreessen Horowitz\ncover design: Jessica Hagy\nproduced using: Pressbooks\nContents\nTHE PMARCA GUIDE TO STARTUPS\nPart 1: Why not to do a startup 2\nPart 2: When the VCs say "no" 10\nPart 3: "But I don\'t know any VCs!" 18\nPart 4: The only thing that matters 25\nPart 5: The Moby Dick theory of big companies 33\nPart 6: How much funding is too little? Too much? 41\nPart 7: Why a startup\'s initial business plan doesn\'t\nmatter that much\n49\nTHE PMARCA GUIDE TO HIRING\nPart 8: Hiring, managing, promoting, and Dring\nexecutives\n54\nPart 9: How to hire a professional CEO 68\nHow to hire the best people you\'ve ever worked\nwith\n69\nTHE PMARCA GUIDE TO BIG COMPANIES\nPart 1: Turnaround! 82\nPart 2: Retaining great people 86\nTHE PMARCA GUIDE TO CAREER, PRODUCTIVITY,\nAND SOME OTHER THINGS\nIntroduction 97\nPart 1: Opportunity 99\nPart 2: Skills and education 107\nPart 3: Where to go and wh

## Task 3: Embeddings and Vectors

Next, we have to convert our corpus into a "machine readable" format as we explored in the Embedding Primer notebook.

Today, we're going to talk about the actual process of creating, and then storing, these embeddings, and how we can leverage that to intelligently add context to our queries.

### OpenAI API Key

In order to access OpenAI's APIs, we'll need to provide our OpenAI API Key!

You can work through the folder "OpenAI API Key Setup" for more information on this process if you don't already have an API Key!

In [60]:
import os
import openai
from getpass import getpass

openai.api_key = getpass("OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai.api_key

### Vector Database

Let's set up our vector database to hold all our documents and their embeddings!

While this is all baked into 1 call - we can look at some of the code that powers this process to get a better understanding:

Let's look at our `VectorDatabase().__init__()`:

```python
def __init__(self, embedding_model: EmbeddingModel = None):
        self.vectors = defaultdict(np.array)
        self.embedding_model = embedding_model or EmbeddingModel()
```

As you can see - our vectors are merely stored as a dictionary of `np.array` objects.

Secondly, our `VectorDatabase()` has a default `EmbeddingModel()` which is a wrapper for OpenAI's `text-embedding-3-small` model.

> **Quick Info About `text-embedding-3-small`**:
> - It has a context window of **8191** tokens
> - It returns vectors with dimension **1536**

#### ❓Question #1:

The default embedding dimension of `text-embedding-3-small` is 1536, as noted above. 

1. Is there any way to modify this dimension?
2. What technique does OpenAI use to achieve this?

> NOTE: Check out this [API documentation](https://platform.openai.com/docs/api-reference/embeddings/create) for the answer to question #1.1, and [this documentation](https://platform.openai.com/docs/guides/embeddings/use-cases) for an answer to question #1.2!


##### ✅ Answer:

**1. Can you modify the embedding dimension?**
Yes! We can actually change the dimension when you call the API. For `text-embedding-3-small`, we can pick any dimension between 1 and 1536. 

Here's how you'd do it:
```python
response = client.embeddings.create(
    input="Your text here",
    model="text-embedding-3-small",
    dimensions=512  # Choose your own size!
)
```

**2. What technique does OpenAI use?**
OpenAI uses something called **"dimension reduction"** or **"truncation"**. Basically, they train the model to work with the full 1536 dimensions, but then they can just chop off the extra dimensions when you ask for a smaller size. 

It's like having a really detailed picture and then cropping it to focus on the most important parts. The model is smart enough that even with fewer dimensions, it still captures the essential meaning of your text. 

We can call the `async_get_embeddings` method of our `EmbeddingModel()` on a list of `str` and receive a list of `float` back!

```python
async def async_get_embeddings(self, list_of_text: List[str]) -> List[List[float]]:
        return await aget_embeddings(
            list_of_text=list_of_text, engine=self.embeddings_model_name
        )
```

We cast those to `np.array` when we build our `VectorDatabase()`:

```python
async def abuild_from_list(self, list_of_text: List[str]) -> "VectorDatabase":
        embeddings = await self.embedding_model.async_get_embeddings(list_of_text)
        for text, embedding in zip(list_of_text, embeddings):
            self.insert(text, np.array(embedding))
        return self
```

And that's all we need to do!

In [8]:
vector_db = VectorDatabase()
vector_db = asyncio.run(vector_db.abuild_from_list(split_documents))

#### ❓Question #2:

What are the benefits of using an `async` approach to collecting our embeddings?

> NOTE: Determining the core difference between `async` and `sync` will be useful! If you get stuck - ask ChatGPT!

##### ✅ Answer:

**Key Benefits of Using Async for Embedding Collection:**

1. **Concurrent API Calls**: The async approach allows multiple embedding requests to be processed simultaneously rather than one at a time. Looking at the code, `async_get_embeddings()` uses `asyncio.gather()` to process multiple batches concurrently.

2. **Significant Performance Improvement**: When processing many documents (like the PMarca blogs), async can be 3-10x faster than synchronous processing because:
   - Network I/O operations (API calls) don't block the entire process
   - Multiple requests can be "in flight" simultaneously
   - CPU can work on other tasks while waiting for API responses

3. **Better Resource Utilization**: 
   - **Sync approach**: CPU sits idle while waiting for each API response
   - **Async approach**: CPU can process other tasks while network requests are pending

4. **Scalability**: As the number of documents increases, the performance gap between sync and async becomes more pronounced. For large document collections, async is essential for reasonable processing times.

5. **Non-blocking Operations**: The async approach prevents the entire application from freezing while waiting for API responses, which is especially important in interactive environments like Jupyter notebooks.

**Core Difference**: 
- **Sync**: Sequential processing - each API call must complete before the next one starts
- **Async**: Concurrent processing - multiple API calls can be initiated and processed simultaneously


So, to review what we've done so far in natural language:

1. We load source documents
2. We split those source documents into smaller chunks (documents)
3. We send each of those documents to the `text-embedding-3-small` OpenAI API endpoint
4. We store each of the text representations with the vector representations as keys/values in a dictionary

### Semantic Similarity

The next step is to be able to query our `VectorDatabase()` with a `str` and have it return to us vectors and text that is most relevant from our corpus.

We're going to use the following process to achieve this in our toy example:

1. We need to embed our query with the same `EmbeddingModel()` as we used to construct our `VectorDatabase()`
2. We loop through every vector in our `VectorDatabase()` and use a distance measure to compare how related they are
3. We return a list of the top `k` closest vectors, with their text representations

There's some very heavy optimization that can be done at each of these steps - but let's just focus on the basic pattern in this notebook.

> We are using [cosine similarity](https://www.engati.com/glossary/cosine-similarity) as a distance metric in this example - but there are many many distance metrics you could use - like [these](https://flavien-vidal.medium.com/similarity-distances-for-natural-language-processing-16f63cd5ba55)

> We are using a rather inefficient way of calculating relative distance between the query vector and all other vectors - there are more advanced approaches that are much more efficient, like [ANN](https://towardsdatascience.com/comprehensive-guide-to-approximate-nearest-neighbors-algorithms-8b94f057d6b6)

In [61]:
vector_db.search_by_text("What is the Michael Eisner Memorial Weak Executive Problem?", k=3)

[('ordingly.\nSeventh, when hiring the executive to run your former specialty, be\ncareful you don’t hire someone weak on purpose.\nThis sounds silly, but you wouldn’t believe how oaen it happens.\nThe CEO who used to be a product manager who has a weak\nproduct management executive. The CEO who used to be in\nsales who has a weak sales executive. The CEO who used to be\nin marketing who has a weak marketing executive.\nI call this the “Michael Eisner Memorial Weak Executive Problem” — aaer the CEO of Disney who had previously been a brilliant TV network executive. When he bought ABC at Disney, it\npromptly fell to fourth place. His response? “If I had an extra\ntwo days a week, I could turn around ABC myself.” Well, guess\nwhat, he didn’t have an extra two days a week.\nA CEO — or a startup founder — oaen has a hard time letting\ngo of the function that brought him to the party. The result: you\nhire someone weak into the executive role for that function so\nthat you can continue to b

## Task 4: Prompts

In the following section, we'll be looking at the role of prompts - and how they help us to guide our application in the right direction.

In this notebook, we're going to rely on the idea of "zero-shot in-context learning".

This is a lot of words to say: "We will ask it to perform our desired task in the prompt, and provide no examples."

### XYZRolePrompt

Before we do that, let's stop and think a bit about how OpenAI's chat models work.

We know they have roles - as is indicated in the following API [documentation](https://platform.openai.com/docs/api-reference/chat/create#chat/create-messages)

There are three roles, and they function as follows (taken directly from [OpenAI](https://platform.openai.com/docs/guides/gpt/chat-completions-api)):

- `{"role" : "system"}` : The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
- `{"role" : "user"}` : The user messages provide requests or comments for the assistant to respond to.
- `{"role" : "assistant"}` : Assistant messages store previous assistant responses, but can also be written by you to give examples of desired behavior.

The main idea is this:

1. You start with a system message that outlines how the LLM should respond, what kind of behaviours you can expect from it, and more
2. Then, you can provide a few examples in the form of "assistant"/"user" pairs
3. Then, you prompt the model with the true "user" message.

In this example, we'll be forgoing the 2nd step for simplicities sake.

#### Utility Functions

You'll notice that we're using some utility functions from the `aimakerspace` module - let's take a peek at these and see what they're doing!

##### XYZRolePrompt

Here we have our `system`, `user`, and `assistant` role prompts.

Let's take a peek at what they look like:

```python
class BasePrompt:
    def __init__(self, prompt):
        """
        Initializes the BasePrompt object with a prompt template.

        :param prompt: A string that can contain placeholders within curly braces
        """
        self.prompt = prompt
        self._pattern = re.compile(r"\{([^}]+)\}")

    def format_prompt(self, **kwargs):
        """
        Formats the prompt string using the keyword arguments provided.

        :param kwargs: The values to substitute into the prompt string
        :return: The formatted prompt string
        """
        matches = self._pattern.findall(self.prompt)
        return self.prompt.format(**{match: kwargs.get(match, "") for match in matches})

    def get_input_variables(self):
        """
        Gets the list of input variable names from the prompt string.

        :return: List of input variable names
        """
        return self._pattern.findall(self.prompt)
```

Then we have our `RolePrompt` which laser focuses us on the role pattern found in most API endpoints for LLMs.

```python
class RolePrompt(BasePrompt):
    def __init__(self, prompt, role: str):
        """
        Initializes the RolePrompt object with a prompt template and a role.

        :param prompt: A string that can contain placeholders within curly braces
        :param role: The role for the message ('system', 'user', or 'assistant')
        """
        super().__init__(prompt)
        self.role = role

    def create_message(self, **kwargs):
        """
        Creates a message dictionary with a role and a formatted message.

        :param kwargs: The values to substitute into the prompt string
        :return: Dictionary containing the role and the formatted message
        """
        return {"role": self.role, "content": self.format_prompt(**kwargs)}
```

We'll look at how the `SystemRolePrompt` is constructed to get a better idea of how that extension works:

```python
class SystemRolePrompt(RolePrompt):
    def __init__(self, prompt: str):
        super().__init__(prompt, "system")
```

That pattern is repeated for our `UserRolePrompt` and our `AssistantRolePrompt` as well.

##### ChatOpenAI

Next we have our model, which is converted to a format analagous to libraries like LangChain and LlamaIndex.

Let's take a peek at how that is constructed:

```python
class ChatOpenAI:
    def __init__(self, model_name: str = "gpt-4.1-mini"):
        self.model_name = model_name
        self.openai_api_key = os.getenv("OPENAI_API_KEY")
        if self.openai_api_key is None:
            raise ValueError("OPENAI_API_KEY is not set")

    def run(self, messages, text_only: bool = True):
        if not isinstance(messages, list):
            raise ValueError("messages must be a list")

        openai.api_key = self.openai_api_key
        response = openai.ChatCompletion.create(
            model=self.model_name, messages=messages
        )

        if text_only:
            return response.choices[0].message.content

        return response
```

#### ❓ Question #3:

When calling the OpenAI API - are there any ways we can achieve more reproducible outputs?

> NOTE: Check out [this section](https://platform.openai.com/docs/guides/text-generation/) of the OpenAI documentation for the answer!

##### ✅ Answer:

**Yes, there are several ways to achieve more reproducible outputs when calling the OpenAI API:**

1. **Temperature Parameter**: 
   - Set `temperature=0` for the most deterministic outputs
   - Lower values (0.0-0.3) = more focused and deterministic
   - Higher values (0.7-1.0) = more creative and random
   - **For reproducibility**: Use `temperature=0`

2. **Seed Parameter** (Newer models):
   - Use the `seed` parameter to get deterministic outputs
   - Same seed + same prompt = same output
   - Example: `seed=42` will always produce the same result for identical inputs

3. **Top_p Parameter**:
   - Controls nucleus sampling (alternative to temperature)
   - Set `top_p=1` for maximum determinism
   - Lower values (0.1-0.5) = more focused responses

4. **Best Practices for Reproducibility**:
   ```python
   response = openai.ChatCompletion.create(
       model="gpt-4",
       messages=[{"role": "user", "content": "Your prompt"}],
       temperature=0,        # Most deterministic
       seed=42,             # Fixed seed for reproducibility
       top_p=1,             # Use all tokens
       max_tokens=150       # Fixed length
   )
   ```

5. **Important Notes**:
   - **Temperature=0** is the most reliable for reproducibility
   - **Seed parameter** works best with temperature=0 or very low values
   - Some randomness may still occur due to model updates or infrastructure changes
   - For production systems, consider caching responses for true reproducibility


### Creating and Prompting OpenAI's `gpt-4.1-mini`!

Let's tie all these together and use it to prompt `gpt-4.1-mini`!

In [62]:
from aimakerspace.openai_utils.prompts import (
    UserRolePrompt,
    SystemRolePrompt,
    AssistantRolePrompt,
)

from aimakerspace.openai_utils.chatmodel import ChatOpenAI

chat_openai = ChatOpenAI()
user_prompt_template = "{content}"
user_role_prompt = UserRolePrompt(user_prompt_template)
system_prompt_template = (
    "You are an expert in {expertise}, you always answer in a kind way."
)
system_role_prompt = SystemRolePrompt(system_prompt_template)

messages = [
    system_role_prompt.create_message(expertise="Python"),
    user_role_prompt.create_message(
        content="What is the best way to write a loop?"
    ),
]

response = chat_openai.run(messages)

In [63]:
print(response)

Hello! The "best" way to write a loop in Python often depends on the specific task you're trying to accomplish, but generally, using a `for` loop is considered the most Pythonic and readable way to iterate over sequences like lists, strings, or ranges.

Here's a simple example of a `for` loop iterating over a list:

```python
fruits = ['apple', 'banana', 'cherry']
for fruit in fruits:
    print(fruit)
```

This loop will print each fruit in the list.

If you want to repeat something a specific number of times, you can use `range()` with a `for` loop:

```python
for i in range(5):
    print(f'Iteration {i}')
```

This will print the iteration number from 0 to 4.

Alternatively, a `while` loop is useful when you want to repeat as long as a condition is true:

```python
count = 0
while count < 5:
    print(f'Count is {count}')
    count += 1
```

Would you like me to help with a particular type of loop or a specific use case?


## Task 5: Retrieval Augmented Generation

Now we can create a RAG prompt - which will help our system behave in a way that makes sense!

There is much you could do here, many tweaks and improvements to be made!

In [64]:
RAG_SYSTEM_TEMPLATE = """You are a knowledgeable assistant that answers questions based strictly on provided context.

Instructions:
- Only answer questions using information from the provided context
- If the context doesn't contain relevant information, respond with "I don't know"
- Be accurate and cite specific parts of the context when possible
- Keep responses {response_style} and {response_length}
- Only use the provided context. Do not use external knowledge.
- Only provide answers when you are confident the context supports your response."""

RAG_USER_TEMPLATE = """Context Information:
{context}

Number of relevant sources found: {context_count}
{similarity_scores}

Question: {user_query}

Please provide your answer based solely on the context above."""

rag_system_prompt = SystemRolePrompt(
    RAG_SYSTEM_TEMPLATE,
    strict=True,
    defaults={
        "response_style": "concise",
        "response_length": "brief"
    }
)

rag_user_prompt = UserRolePrompt(
    RAG_USER_TEMPLATE,
    strict=True,
    defaults={
        "context_count": "",
        "similarity_scores": ""
    }
)

Now we can create our pipeline!

In [65]:
class RetrievalAugmentedQAPipeline:
    def __init__(self, llm: ChatOpenAI, vector_db_retriever: VectorDatabase, 
                 response_style: str = "detailed", include_scores: bool = False) -> None:
        self.llm = llm
        self.vector_db_retriever = vector_db_retriever
        self.response_style = response_style
        self.include_scores = include_scores

    def run_pipeline(self, user_query: str, k: int = 4, **system_kwargs) -> dict:
        # Retrieve relevant contexts
        context_list = self.vector_db_retriever.search_by_text(user_query, k=k)
        
        context_prompt = ""
        similarity_scores = []
        
        for i, (context, score) in enumerate(context_list, 1):
            context_prompt += f"[Source {i}]: {context}\n\n"
            similarity_scores.append(f"Source {i}: {score:.3f}")
        
        # Create system message with parameters
        system_params = {
            "response_style": self.response_style,
            "response_length": system_kwargs.get("response_length", "detailed")
        }
        
        formatted_system_prompt = rag_system_prompt.create_message(**system_params)
        
        user_params = {
            "user_query": user_query,
            "context": context_prompt.strip(),
            "context_count": len(context_list),
            "similarity_scores": f"Relevance scores: {', '.join(similarity_scores)}" if self.include_scores else ""
        }
        
        formatted_user_prompt = rag_user_prompt.create_message(**user_params)

        return {
            "response": self.llm.run([formatted_system_prompt, formatted_user_prompt]), 
            "context": context_list,
            "context_count": len(context_list),
            "similarity_scores": similarity_scores if self.include_scores else None,
            "prompts_used": {
                "system": formatted_system_prompt,
                "user": formatted_user_prompt
            }
        }

In [66]:
rag_pipeline = RetrievalAugmentedQAPipeline(
    vector_db_retriever=vector_db,
    llm=chat_openai,
    response_style="detailed",
    include_scores=True
)

result = rag_pipeline.run_pipeline(
    "What is the 'Michael Eisner Memorial Weak Executive Problem'?",
    k=3,
    response_length="comprehensive", 
    include_warnings=True,
    confidence_required=True
)

print(f"Response: {result['response']}")
print(f"\nContext Count: {result['context_count']}")
print(f"Similarity Scores: {result['similarity_scores']}")

Response: The "Michael Eisner Memorial Weak Executive Problem" refers to a common issue where a CEO or startup founder hires a weak executive to run the function that originally brought the CEO or founder to prominence, in order to continue being "the man" in that area. The problem is named after Michael Eisner, the CEO of Disney who previously was a brilliant TV network executive. When he bought ABC at Disney, the network promptly fell to fourth place, and Eisner responded by saying that if he had extra time, he could personally turn ABC around. However, he did not have that extra time. This situation exemplifies how CEOs often have difficulty letting go of their former specialty and consequently hire weak executives for those roles, which can negatively impact the organization's performance. This explanation is drawn from Source 1, which states:

"I call this the 'Michael Eisner Memorial Weak Executive Problem' — after the CEO of Disney who had previously been a brilliant TV network 

#### ❓ Question #4:

What prompting strategies could you use to make the LLM have a more thoughtful, detailed response?

What is that strategy called?

> NOTE: You can look through our [OpenAI Responses API](https://colab.research.google.com/drive/14SCfRnp39N7aoOx8ZxadWb0hAqk4lQdL?usp=sharing) notebook for an answer to this question if you get stuck!

##### ✅ Answer:

**Several prompting strategies can make the LLM have more thoughtful, detailed responses:**

1. **Chain of Thought (CoT) Prompting** - This is the main strategy:
   - Ask the model to "think step by step" or "show your reasoning"
   - Example: "Let's solve this step by step. First, I need to..."
   - Forces the model to break down complex problems into smaller parts
   - Leads to more thorough and logical responses

2. **Few-Shot Examples with Reasoning**:
   - Provide examples that show the desired reasoning process
   - Include both the problem and the step-by-step solution
   - Helps the model understand the expected thought process

3. **Explicit Instruction for Detail**:
   - "Please provide a detailed explanation"
   - "Explain your reasoning behind each step"
   - "Consider multiple perspectives before answering"

4. **Role-Based Prompting**:
   - "You are an expert in [field] who explains concepts clearly"
   - "Act as a teacher who breaks down complex topics"

5. **Question Decomposition**:
   - "First, let's identify the key components..."
   - "What are the main factors to consider?"
   - "How would you approach this systematically?"

**The main strategy is called "Chain of Thought (CoT) Prompting"**

**Example of CoT in practice:**
```
"Let's think about this step by step:
1. First, I need to identify the main components
2. Then, I'll analyze each component
3. Finally, I'll synthesize the findings"
```

This approach significantly improves the quality and depth of LLM responses by encouraging systematic thinking rather than jumping to conclusions.


### 🏗️ Activity #1:

Enhance your RAG application in some way! 

Suggestions are: 

- Allow it to work with PDF files
- Implement a new distance metric
- Add metadata support to the vector database
- Use a different embedding model
- Add the capability to ingest a YouTube link

While these are suggestions, you should feel free to make whatever augmentations you desire! If you shared an idea during Session 1, think about features you might need to incorporate for your use case! 

When you're finished making the augments to your RAG application - vibe check it against the old one - see if you can "feel the improvement"!

> NOTE: These additions might require you to work within the `aimakerspace` library - that's expected!

> NOTE: If you're not sure where to start - ask Cursor (CMD/CTRL+L) to guide you through the changes!

In [73]:
### YOUR CODE HERE

# Simple Enhanced RAG Application - Beginner Friendly!

import numpy as np
from collections import defaultdict
from typing import List, Tuple, Callable, Dict, Any
import uuid
from datetime import datetime

# Import the cosine_similarity function from the existing codebase
from aimakerspace.vectordatabase import cosine_similarity

# 1. Simple Enhanced Vector Database with Basic Metadata
class EnhancedVectorDatabase:
    def __init__(self, embedding_model=None):
        self.vectors = defaultdict(np.array)
        self.metadata = defaultdict(dict)
        self.embedding_model = embedding_model or EmbeddingModel()
    
    def insert(self, key: str, vector: np.array, metadata: Dict[str, Any] = None):
        """Insert a vector with optional metadata"""
        self.vectors[key] = vector
        self.metadata[key] = metadata or {}
        # Add automatic metadata
        if 'id' not in self.metadata[key]:
            self.metadata[key]['id'] = str(uuid.uuid4())
        if 'created_at' not in self.metadata[key]:
            self.metadata[key]['created_at'] = datetime.now().isoformat()
    
    def search(self, query_vector: np.array, k: int, distance_measure: Callable = cosine_similarity) -> List[Tuple[str, float, Dict]]:
        """Search with metadata included in results"""
        scores = [
            (key, distance_measure(query_vector, vector), self.metadata[key])
            for key, vector in self.vectors.items()
        ]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:k]
    
    def search_by_text(self, query_text: str, k: int, distance_measure: Callable = cosine_similarity) -> List[Tuple[str, float, Dict]]:
        """Search by text with metadata"""
        query_vector = self.embedding_model.get_embedding(query_text)
        return self.search(query_vector, k, distance_measure)
    

    
    async def abuild_from_list(self, list_of_text: List[str], metadata_list: List[Dict] = None) -> "EnhancedVectorDatabase":
        """Build database from text list with optional metadata"""
        embeddings = await self.embedding_model.async_get_embeddings(list_of_text)
        for i, (text, embedding) in enumerate(zip(list_of_text, embeddings)):
            metadata = metadata_list[i] if metadata_list and i < len(metadata_list) else {}
            # Add chunk information
            metadata.update({
                'chunk_index': i,
                'text_length': len(text),
                'source': 'PMarcaBlogs.txt'
            })
            self.insert(text, np.array(embedding), metadata)
        return self
    


# 2. New Distance Metric (Simple Addition)
def euclidean_distance(vector_a: np.array, vector_b: np.array) -> float:
    """Euclidean distance - measures straight-line distance between vectors"""
    return -np.linalg.norm(vector_a - vector_b)

# 3. Simple Enhanced RAG Pipeline
class EnhancedRAGPipeline:
    def __init__(self, llm: ChatOpenAI, vector_db_retriever: EnhancedVectorDatabase, 
                 response_style: str = "detailed", include_scores: bool = False,
                 distance_metric: str = "cosine") -> None:
        self.llm = llm
        self.vector_db_retriever = vector_db_retriever
        self.response_style = response_style
        self.include_scores = include_scores
        
        # Simple distance metric mapping
        self.distance_metrics = {
            "cosine": cosine_similarity,
            "euclidean": euclidean_distance
        }
        self.distance_metric = self.distance_metrics.get(distance_metric, cosine_similarity)

    def run_pipeline(self, user_query: str, k: int = 4, **system_kwargs) -> dict:
        # Retrieve relevant contexts with metadata
        context_list = self.vector_db_retriever.search_by_text(
            user_query, k=k, distance_measure=self.distance_metric
        )
        
        context_prompt = ""
        similarity_scores = []
        metadata_info = []
        
        for i, (context, score, metadata) in enumerate(context_list, 1):
            context_prompt += f"[Source {i}]: {context}\n\n"
            similarity_scores.append(f"Source {i}: {score:.3f}")
            metadata_info.append({
                'source_id': i,
                'chunk_index': metadata.get('chunk_index', 'N/A'),
                'text_length': metadata.get('text_length', 'N/A'),
                'source_file': metadata.get('source', 'N/A')
            })
        
        # Create system message with parameters
        system_params = {
            "response_style": self.response_style,
            "response_length": system_kwargs.get("response_length", "detailed")
        }
        
        formatted_system_prompt = rag_system_prompt.create_message(**system_params)
        
        user_params = {
            "user_query": user_query,
            "context": context_prompt.strip(),
            "context_count": len(context_list),
            "similarity_scores": f"Relevance scores: {', '.join(similarity_scores)}" if self.include_scores else ""
        }
        
        formatted_user_prompt = rag_user_prompt.create_message(**user_params)

        return {
            "response": self.llm.run([formatted_system_prompt, formatted_user_prompt]), 
            "context": context_list,
            "context_count": len(context_list),
            "similarity_scores": similarity_scores if self.include_scores else None,
            "metadata": metadata_info,
            "distance_metric_used": self.distance_metric.__name__,
            "prompts_used": {
                "system": formatted_system_prompt,
                "user": formatted_user_prompt
            }
        }

# 4. Build Simple Enhanced Vector Database
print("Building simple enhanced vector database with metadata...")
enhanced_vector_db = EnhancedVectorDatabase()
enhanced_vector_db = asyncio.run(enhanced_vector_db.abuild_from_list(split_documents))

print(f"Enhanced database built with {len(enhanced_vector_db.vectors)} documents")
print(f"Sample metadata: {list(enhanced_vector_db.metadata.values())[0]}")

# 5. Test Simple Enhanced RAG Pipeline
print("\n" + "="*50)
print("TESTING ENHANCED RAG PIPELINE")
print("="*50)

# Test with cosine similarity (original)
print("\n1. Testing with Cosine Similarity (Original):")
enhanced_rag_cosine = EnhancedRAGPipeline(
    vector_db_retriever=enhanced_vector_db,
    llm=chat_openai,
    response_style="detailed",
    include_scores=True,
    distance_metric="cosine"
)

result_cosine = enhanced_rag_cosine.run_pipeline(
    "What is the 'Michael Eisner Memorial Weak Executive Problem'?",
    k=3
)

print(f"Response: {result_cosine['response'][:200]}...")
print(f"Distance metric: {result_cosine['distance_metric_used']}")
print(f"Metadata sample: {result_cosine['metadata'][0]}")

# Test with Euclidean distance (new)
print("\n2. Testing with Euclidean Distance (New):")
enhanced_rag_euclidean = EnhancedRAGPipeline(
    vector_db_retriever=enhanced_vector_db,
    llm=chat_openai,
    response_style="detailed",
    include_scores=True,
    distance_metric="euclidean"
)

result_euclidean = enhanced_rag_euclidean.run_pipeline(
    "What is the 'Michael Eisner Memorial Weak Executive Problem'?",
    k=3
)

print(f"Response: {result_euclidean['response'][:200]}...")
print(f"Distance metric: {result_euclidean['distance_metric_used']}")
print(f"Metadata sample: {result_euclidean['metadata'][0]}")

# 6. Compare Results
print("\n" + "="*50)
print("COMPARISON SUMMARY")
print("="*50)
print(f"Cosine Similarity Results: {len(result_cosine['context'])} sources found")
print(f"Euclidean Distance Results: {len(result_euclidean['context'])} sources found")
print(f"Enhanced features: Metadata tracking, Multiple distance metrics, Better document management")

print("\n✅ Simple Enhancement Complete!")
print("Key improvements:")
print("- Added metadata support for better document tracking")
print("- Implemented new distance metric (euclidean)")
print("- Enhanced pipeline with metadata information")
print("- Maintained backward compatibility with original system")
print("- Now you can see exactly which chunk was used!")
print("- Simple and easy to understand!")

Building simple enhanced vector database with metadata...
Enhanced database built with 373 documents
Sample metadata: {'chunk_index': 0, 'text_length': 1000, 'source': 'PMarcaBlogs.txt', 'id': 'c7e9fa05-8f77-4534-9b98-4ea06940d966', 'created_at': '2025-09-15T19:11:44.991711'}

TESTING ENHANCED RAG PIPELINE

1. Testing with Cosine Similarity (Original):
Response: The "Michael Eisner Memorial Weak Executive Problem" refers to the phenomenon where a CEO who used to be deeply involved and skilled in a particular function (such as product management, sales, or mar...
Distance metric: cosine_similarity
Metadata sample: {'source_id': 1, 'chunk_index': 115, 'text_length': 1000, 'source_file': 'PMarcaBlogs.txt'}

2. Testing with Euclidean Distance (New):
Response: The "Michael Eisner Memorial Weak Executive Problem" refers to a phenomenon where a CEO or startup founder, who previously excelled in a particular function (such as product management, sales, or mark...
Distance metric: euclidean_dis